# 06 — Logfire: see what your agent did

**Tooling** · optional

> **This notebook is optional.** Nothing later depends on it. If you don't want a Logfire account, skip to
> [07_history_and_streaming.ipynb](07_history_and_streaming.ipynb). Every cell here still runs without an
> account; the traces just aren't sent anywhere.

[Logfire](https://logfire.pydantic.dev) is Pydantic's observability tool. Each agent run becomes a
**trace**: a tree of timed steps, called **spans**, one for each model call and each tool call, with the
arguments, results, tokens and cost.

When you finish you can:

- send this course's traces to your own Logfire project
- read an agent's trace in the Live view
- name your agents and add spans of your own
- find a failed tool call (a `ModelRetry`) in a trace

In [ ]:
from learningdantic import setup

MODEL = setup()

## Step 1: connect a Logfire project (once)

`setup()` already turned Logfire on, but it only sends data when it finds credentials for a project.
Pick one route.

**Route A — a write token in `.env` (simplest)**

1. Sign in at [logfire.pydantic.dev](https://logfire.pydantic.dev) and create a project.
2. In the project, open **Settings → Write tokens** and create a token.
3. Add one line to the `.env` file at the repo root: `LOGFIRE_TOKEN=<your token>`.

**Route B — the Logfire CLI**, in a terminal at the repo root:

```bash
uv run logfire auth            # log in through the browser (on its own, this sends nothing)
uv run logfire projects new    # create a project; saves its credentials in .logfire/ at the repo root
```

(Already have a project? Use `uv run logfire projects use <name>` instead of `projects new`.)

The notebook runs from `notebooks/`, not the repo root. `setup()` handles that: it sets
`LOGFIRE_CREDENTIALS_DIR` to the repo root's `.logfire/` folder, so Logfire finds those credentials anyway.

Either way, then restart the kernel (**Kernel → Restart**) and run the setup cell again: Logfire reads
credentials once, when it starts. This cell tells you which case you're in:

In [ ]:
import os
from pathlib import Path

credentials = Path(os.environ["LOGFIRE_CREDENTIALS_DIR"]) / "logfire_credentials.json"
if os.getenv("LOGFIRE_TOKEN") or credentials.exists():
    print("✅ Logfire credentials found: traces from this notebook will be sent.")
else:
    print("No Logfire credentials yet: every cell still runs, but nothing is sent. See Step 1.")

> ✅ **Checkpoint** — one of those two lines. Either way, carry on: the cells below work with or without
> Logfire.

## Step 2: a named agent and a span of your own

Two additions make traces easier to read:

- `Agent(..., name="inventory_agent")` labels every run of this agent.
- `with logfire.span("checking stock for {sku}", sku=sku):` opens a span of your own. A `with` block runs
  its body and then closes the span, so the span times exactly that body. `{sku}` in the message is filled
  in from the `sku=` value.

Open your project's **Live view** next to this notebook, then run:

In [ ]:
import logfire
from pydantic_ai import Agent

STOCK = {"ABC-1": 42, "GIZMO-9": 7, "WIDGET-3": 0}

inventory_agent = Agent(
    MODEL,
    name="inventory_agent",
    instructions="You manage a tiny inventory. Use your tool for every stock number; never invent one.",
)


@inventory_agent.tool_plain
def lookup_stock(sku: str) -> str:
    """Look up the current stock for a SKU."""
    with logfire.span("checking stock for {sku}", sku=sku):
        quantity = STOCK.get(sku.upper())
    return "unknown SKU" if quantity is None else f"{quantity} in stock"


r = await inventory_agent.run("How many ABC-1 and WIDGET-3 do we have? What is the total?")
print(r.output)

> ✅ **Checkpoint** — something like `ABC-1: 42 in stock. WIDGET-3: 0 in stock. **Total: 42.**`

> 🔭 **In Logfire** — the Live view shows one trace for the run, roughly:
>
> ```
> inventory_agent run
> ├── chat <model name>              the model asks for the tool (tokens and cost on this line)
> ├── running tool: lookup_stock
> │   └── checking stock for ABC-1    your span, with sku as an attribute
> ├── running tool: lookup_stock
> │   └── checking stock for WIDGET-3
> └── chat <model name>              the model writes the answer
> ```
>
> `chat` lines show your MODEL's name without the `openai:` part. Click any line to see its arguments,
> result and timing. The number of `chat` lines can vary from run to run.

## Step 3: catch a failing tool

The tool below fails on its first call with a `ModelRetry` (you met it in notebook 04), then works. The
counter is a dict, so the tool can update it. This cell builds its own agent and counter, so you can run
it as often as you like.

In [ ]:
from pydantic_ai import ModelRetry

calls = {"fetch_price": 0}

price_agent = Agent(
    MODEL,
    name="price_agent",
    instructions="Answer price questions with the fetch_price tool. Never guess a price.",
)


@price_agent.tool_plain
def fetch_price(sku: str) -> str:
    """Fetch the current price of a SKU."""
    calls["fetch_price"] += 1
    if calls["fetch_price"] == 1:
        raise ModelRetry("Price service hiccup. Call fetch_price again.")
    return "$9.99"


r = await price_agent.run("What does ABC-1 cost?")
print(r.output)
print("fetch_price was called", calls["fetch_price"], "times")

> ✅ **Checkpoint** — `ABC-1 costs $9.99.` (or similar), then `fetch_price was called 2 times`.

> 🔭 **In Logfire** — the first `running tool: fetch_price` line is marked as an error and carries your
> message. The next `chat` line is the model reading that message, and the second
> `running tool: fetch_price` succeeds. That's the everyday debugging move: the trace shows which tool
> failed, and with what arguments, without a single `print`.

## What `setup()` did for you

In your own projects, these are the two lines that turn Logfire on. `setup()` runs them for you:

```python
import logfire

logfire.configure(send_to_logfire="if-token-present", console=False)  # send only if credentials exist
logfire.instrument_pydantic_ai()  # every agent run becomes a trace
```

Before that, `setup()` set `LOGFIRE_CREDENTIALS_DIR` to the repo root's `.logfire/` folder, which is how
`logfire.configure` finds Route B's credentials from inside `notebooks/`. In your own project, run your
code from the folder that holds `.logfire/`, or use a `LOGFIRE_TOKEN`. (`setup()` also passes `distributed_tracing=True`,
so notebook 09's in-process MCP server joins the trace quietly; you only need that with MCP.)

## The two dashboards

- **Agents view**: one row per agent (that's what `name=` is for): runs, cost, duration, turns per run.
- **LLMs view**: one row per model: calls, errors, latency, tokens per second, cost.

## 🏋️ Exercise — the joker and the critic

1. Build two named agents. `joker` tells one short programming joke, and its instructions say to **always
   call** its `get_setup_lines` tool first. That tool is a plain `def` that opens a `logfire.span`.
   `critic` rates a joke from 0 to 10 and uses an `output_type` (notebook 03), so the score is an `int`.
2. Run 3 rounds: the joker writes a joke and the critic rates it. If you connected Logfire, open the
   **Agents view** to compare the two agents' runs and cost, and find your span inside one of the joker's
   traces.

Stretch: wrap each round in `with logfire.span("joke round {n}", n=n):`, so both agents' runs sit under
one parent line.

Your take: [`../solutions/06_two_agents_spans.ipynb`](../solutions/06_two_agents_spans.ipynb).

## Next →

→ **[07_history_and_streaming.ipynb](07_history_and_streaming.ipynb)** — conversations that carry on between
runs, and answers that appear as they are written.